In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet101, ResNet101_Weights
import torchvision.transforms as transforms
import gensim.downloader as gensimApi

import nltk
import json
import random
import math
import matplotlib.pyplot as plt
import re
import string
import pickle

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm.auto import tqdm
from typing import Literal

# gensimApi is sooo slow! Save weights directly in my kaggle dataset
# glove = gensimApi.load("glove-wiki-gigaword-300")
# with open("glove.pkl", "wb") as f:
#     pickle.dump(glove, f)

nltk.download('punkt_tab', quiet=True)

def load_json(path: Path | str) -> dict:
    with Path(path).open() as f:
        return json.load(f)

In [ ]:
# VQA

# VQA_IMAGES_TRAIN_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Images/train2014')
# VQA_IMAGES_VAL_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Images/val2014')
# VQA_IMAGES_TEST_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Images/test2015')

VQA_IMAGES_FEAT_TRAIN_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/vqa_image_features_train.pt')
VQA_IMAGES_FEAT_VAL_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/vqa_image_features_val.pt')
VQA_IMAGES_FEAT_TEST_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/vqa_image_features_test.pt')

VQA_QUESTIONS_TRAIN_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Questions/v2_OpenEnded_mscoco_train2014_questions.json')
VQA_QUESTIONS_VAL_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Questions/v2_OpenEnded_mscoco_val2014_questions.json')
VQA_QUESTIONS_TEST_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Questions/v2_OpenEnded_mscoco_test2015_questions.json')

VQA_ANNOTATIONS_TRAIN_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Annotations/v2_mscoco_train2014_annotations.json')
VQA_ANNOTATIONS_VAL_PATH = Path('/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/VQA v2/VQA v2/Annotations/v2_mscoco_val2014_annotations.json')

# taken from vqa eval official code (and saved as json)
_NORM = load_json("/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/vqa_normalization.json")
CONTRACTIONS = _NORM["contractions"]
MANUAL_MAP = _NORM["manualMap"]
ARTICLES = _NORM["articles"]
PUNCT = _NORM["punct"]
PERIOD_STRIP = re.compile(r"(?!<=\d)(\.)(?!\d)")
COMMA_STRIP = re.compile(r"(\d)(\,)(\d)")

# GQA

# GQA_IMAGES_PATH = Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/GQA images/images')

GQA_IMAGES_FEAT_PATH = Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/gqa_image_features.pt')

GQA_QUESTIONS_TRAIN_PATH = Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/GQA questions/GQA questions/train_balanced_questions.json')
GQA_QUESTIONS_VAL_PATH = Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/GQA questions/GQA questions/val_balanced_questions.json')
GQA_QUESTIONS_TESTDEV_PATH = Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/GQA questions/GQA questions/testdev_balanced_questions.json')
GQA_QUESTIONS_TEST_PATH = Path('/kaggle/input/datasets/davideperniconi/gqa-dataset/GQA questions/GQA questions/test_balanced_questions.json')

# ------

IMAGE_NET_DIM = 224
IMAGE_NET_MEAN = [0.485, 0.456, 0.406]
IMAGE_NET_STD = [0.229, 0.224, 0.225]

QUESTION_VOCAB_MAX_SIZE = 7500
VQA_ANSWER_VOCAB_SIZE = 3129
GQA_ANSWER_VOCAB_SIZE = 1842
MAX_SEQ_LEN = 20

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open("/kaggle/input/datasets/davideperniconi/hwp-vqa-v2/glove.pkl", "rb") as f:
    glove = pickle.load(f)

# Compute Image Net embeddings

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path  = self.image_paths[idx]
        image = self.transform(Image.open(path).convert("RGB"))
        filename = path.stem
    
        image_id = filename.split('_')[-1] if filename.startswith("COCO") else filename
        return image, image_id


@torch.inference_mode()
def extract_and_save_features(image_dir: Path, output_path: Path, batch_size: int = 128):
    backbone = resnet101(weights=ResNet101_Weights.DEFAULT)
    backbone = nn.Sequential(*list(backbone.children())[:-1])
    backbone = backbone.to(DEVICE).eval()
    
    if torch.cuda.device_count() > 1:
        backbone = nn.DataParallel(backbone)

    transform = transforms.Compose([
        transforms.Resize((IMAGE_NET_DIM, IMAGE_NET_DIM)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGE_NET_MEAN, std=IMAGE_NET_STD)
    ])

    image_paths = sorted(image_dir.glob("*.jpg"))
    
    dataset = ImageDataset(image_paths, transform)
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)
    features = {}

    for images, image_ids in tqdm(loader, desc=f"Extracting {image_dir.name}"):
        images = images.to(DEVICE)
        
        with torch.autocast(device_type='cuda'):
            feats = backbone(images)

        feats = feats.float().flatten(1)
        #feats = F.normalize(feats, p=2, dim=-1) # the paper does it ("I norm") / I'll decide later if normalize or not in the model
        
        for image_id, feat in zip(image_ids, feats):
            features[image_id] = feat.cpu()

    torch.save(features, output_path)
    
    print(f"Saved {len(features)} features to {output_path}")
    del backbone


# pre-calculate all images features otherwise there is gpu burst problem during training :(
# extract_and_save_features(VQA_IMAGES_TRAIN_PATH, Path("vqa_image_features_train.pt"))
# extract_and_save_features(VQA_IMAGES_VAL_PATH, Path("vqa_image_features_val.pt"))
# extract_and_save_features(VQA_IMAGES_TEST_PATH, Path("vqa_image_features_test.pt"))
#extract_and_save_features(GQA_IMAGES_PATH, Path("gqa_image_features.pt"))

# Pre-processing

In [ ]:
def process_punctuation(text: str) -> str:
    for p in PUNCT:
        if (p + ' ' in text or ' ' + p in text) or COMMA_STRIP.search(text):
            text = text.replace(p, '')
        else:
            text = text.replace(p, ' ')
    
    return PERIOD_STRIP.sub("", text)


def process_digit_article(text: str) -> str:
    words = text.lower().split()
    words = [MANUAL_MAP.get(w, w) for w in words]
    words = [w for w in words if w not in ARTICLES]
    words = [CONTRACTIONS.get(w, w) for w in words]
    return ' '.join(words)


def normalize_answers(answers: list[str]) -> list[str]:
    answers = [a.replace('\n', ' ').replace('\t', ' ').strip() for a in answers]
    
    if len(set(answers)) > 1:
        answers = [process_punctuation(a) for a in answers]
        answers = [process_digit_article(a) for a in answers]
    
    return answers


def load_vqa_data(questions_path: Path, annotations_path: Path):
    questions = load_json(questions_path)['questions']
    annotations = load_json(annotations_path)['annotations']

    annotations_dict = {a['question_id']: a for a in annotations}

    data = []

    for q in questions:
        qid = q['question_id']

        if qid not in annotations_dict:
            continue

        item = {**q, **annotations_dict[qid]}

        answers = normalize_answers([a['answer'] for a in item['answers']])

        data.append({
            "question_id": qid,
            "question": item["question"],
            "image_id": item["image_id"],
            "answers": answers
        })

    return data


def load_vqa_test_data(questions_path: Path):
    questions = load_json(questions_path)['questions']
    data = []

    for q in questions:
        data.append({
            "question_id": q['question_id'],
            "question": q["question"],
            "image_id": q["image_id"],
        })

    return data
    

def load_gqa_data(questions_path: Path) -> list[dict]:
    questions = load_json(questions_path)
    data = []

    for question_id, question in questions.items():
        answer = question['answer'].strip().lower()
        
        data.append({
            "question_id": question_id,
            "question": question['question'],
            "image_id": question['imageId'],
            "answers": [answer],
            'is_binary': answer in {"yes", "no"}
        })

    return data


def build_answers_vocab(data: list[dict], vocab_size: int):
    counter = Counter()

    for question in data:
        counter.update(question['answers'])
        
    most_common = counter.most_common(vocab_size)
                
    return {ans: i for i, (ans, _) in enumerate(most_common)}


def compute_soft_labels(answers: list, vocab: dict) -> torch.Tensor:
    label = torch.zeros(len(vocab))
    counter = Counter(answers)  # already normalized
    
    for answer, count in counter.items():
        if answer in vocab:
            label[vocab[answer]] = min(count / 3, 1.0)
    
    return label


def compute_vqa_soft_labels(vqa_data: list[dict], vqa_dict: dict[str, int]):
    result = []
    
    for q in vqa_data:
        question = {**q}
        soft_labels = compute_soft_labels(q['answers'], vqa_dict)
        
        if soft_labels.sum() == 0:
            continue
        else:
            question['label'] = soft_labels

        result.append(question)

    return result


def compute_gqa_label(gqa_data: list[dict], gqa_dict: dict[str, int]):
    result = []
    
    for q in gqa_data:
        question = {**q}
        answer_idx = gqa_dict.get(q['answers'][0], None)

        if answer_idx is None:
            continue
        else:
            question['label'] = answer_idx

        result.append(question)

    return result

In [ ]:
class QuestionTokenizer:
    def __init__(self, max_size: int | None = None, min_freq: int = 0, special_tokens: list[str] = []):
        self.min_freq = min_freq
        self.max_size = max_size
        self.special_tokens = special_tokens
        self.word2idx: dict[str, int] = {}
        self.idx2word: dict[int, str] = {}
        self.vocab_size: int = 0

    def tokenize(self, text: str) -> list[str]:
        tokens = nltk.word_tokenize(text.lower())
        return [t for t in tokens if t not in string.punctuation]
        
    def build_vocab(self, questions: list[dict]) -> None:
        counter = Counter()
        for q in questions:
            counter.update(self.tokenize(q['question']))
        
        valid_tokens = [w for w, c in counter.items() if c >= self.min_freq]        
        sorted_tokens = sorted(valid_tokens, key=lambda w: counter[w], reverse=True)
        
        if self.max_size:
            limit = self.max_size - len(self.special_tokens)
            sorted_tokens = sorted_tokens[:limit]
        
        all_tokens = self.special_tokens + sorted_tokens
        self.word2idx = {token: i for i, token in enumerate(all_tokens)}
        self.idx2word = {i: token for token, i in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)

    
    def encode_questions(self, questions: list[dict], max_length: int = 15, return_tensors: bool = True) -> list[dict]:
        result = []
        
        for q in questions:
            item = {**q}
            item['question'] = self.encode(q['question'], max_length, return_tensors)
            result.append(item)

        return result
    
    def encode(self, text: str, max_length: int = 15, return_tensors: bool = True) -> list[int] | torch.Tensor:
        tokens = self.tokenize(text)
        unk_idx = self.word2idx.get('<UNK>', 1)
        pad_idx = self.word2idx.get('<PAD>', 0)
        
        encoded = [self.word2idx.get(t, unk_idx) for t in tokens]
        
        if len(encoded) < max_length:
            encoded += [pad_idx] * (max_length - len(encoded))
        else:
            encoded = encoded[:max_length]
            
        if return_tensors:
            return torch.tensor(encoded, dtype=torch.long)
        
        return encoded

    def decode(self, indices: list[int] | torch.Tensor) -> str:
        if torch.is_tensor(indices):
            indices = indices.tolist()

        tokens = []

        for idx in indices:
            word = self.idx2word.get(idx, '<UNK>')

            if word not in self.special_tokens:
                tokens.append(word)
            
        return " ".join(tokens)

    def save(self, path: Path | str) -> None:
        data = {
            'word2idx': self.word2idx,
            'special_tokens': self.special_tokens,
            'config': {'min_freq': self.min_freq, 'max_size': self.max_size}
        }
        Path(path).write_text(json.dumps(data), encoding='utf-8')

    def load(self, path: Path | str) -> None:
        data = load_json(path)
        
        self.word2idx = data['word2idx']
        self.special_tokens = data['special_tokens']
        self.min_freq = data['config']['min_freq']
        self.max_size = data['config']['max_size']
        self.idx2word = {int(k): v for v, k in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)


class HWpDataset(Dataset):
    def __init__(self, data: list, image_features: dict[int, torch.Tensor]):
        self.data = data
        self.image_features = image_features

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        return {
            "image": self.image_features[item['image_id']], # [2048]
            "question": item['question'], # [MAX_LEN]
            "question_id": item['question_id'],
            "label":  item.get('label', None),  # None for test,
            "is_binary":  item.get('is_binary', None)  # Only for gqa
        }


def build_glove_weights(tokenizer: QuestionTokenizer, glove, embed_dim: int = 300) -> torch.Tensor:
    weights = torch.zeros(len(tokenizer.word2idx), embed_dim)
    found = 0
    
    for word, idx in tokenizer.word2idx.items():
        if word in glove:
            weights[idx] = torch.tensor(glove[word], dtype=torch.float)
            found += 1
        elif word not in tokenizer.special_tokens:
            print(f'Word {word} not found')
            weights[idx] = torch.randn(embed_dim) * 0.01

    print(f"GloVe: {found}/{len(tokenizer.word2idx)} words found ({100 * found / len(tokenizer.word2idx):.1f}%)")
    
    return weights

# Vanilla VQA model

In [ ]:
class VanillaVQA(nn.Module):
    def __init__(
        self,
        question_vocab_size: int,
        answer_vocab_size: int,
        dropout: float = 0.2,
        glove_weights: torch.Tensor = None
    ):
        super().__init__()

        embed_dim = 300
        lstm_hidden_dim = 512
        
        # for param in self.image_encoder.parameters():
        #     param.requires_grad = False
                
        if glove_weights is not None:
            self.embedding = nn.Embedding.from_pretrained(
                glove_weights,
                freeze=False,
                padding_idx=0
            )
        else:
            self.embedding = nn.Embedding(question_vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=dropout
        )

        # self.image_proj = nn.Linear(2048, 1024)
        # self.question_proj = nn.Linear(2048, 1024)
        self.image_proj = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.LayerNorm(1024),   # normalizza le scale → prodotto stabile
            nn.Tanh(),            # ← Tanh invece di GELU/ReLU con Hadamard
            nn.Dropout(dropout)
        )
        self.question_proj = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.LayerNorm(1024),
            nn.Tanh(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Sequential(
            nn.Linear(1024, 1024),
            nn.LayerNorm(1024),   # ← stabilizza dopo Hadamard
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, answer_vocab_size)
        )

    def forward(self, image_features: torch.Tensor, question: torch.Tensor) -> torch.Tensor:
        # image branch
        # image_features = self.image_encoder(image) # [B, 2048, 1, 1]
        # image_features = image_features.flatten(1) # [B, 2048]
        image_features = F.normalize(image_features, p=2, dim=-1) # [B, 2048]
        
        # question branch
        question_embed = self.embedding(question)
        _, (hidden, cell) = self.lstm(question_embed)
        question_features = torch.cat([
            hidden[0], cell[0],
            hidden[1], cell[1]
        ], dim=-1) # [B, 2048]
        question_features = F.normalize(question_features, p=2, dim=-1) # ← manca questo

        # MLPs + fusion
        image_features = self.image_proj(image_features)
        question_features = self.question_proj(question_features)

        features = image_features * question_features

        return self.classifier(features)

# Training

In [ ]:
def compute_loss(logits, label):
    if label.dim() == 2:
        # VQA — soft labels [B, vocab_size] → BCE
        return F.binary_cross_entropy_with_logits(logits, label)
    
    # GQA — hard labels [B] → Cross Entropy
    return F.cross_entropy(logits, label)

def compute_acc(logits, label):
    preds = logits.argmax(dim=-1)
    
    if label.dim() == 2: # VQA — soft accuracy
        return label[torch.arange(preds.size(0)), preds].sum().item()
    
    # GQA — hard accuracy
    return (preds == label).sum().item()


def train_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc="Training")
    
    for batch in pbar:
        images = batch['image'].to(DEVICE)
        questions = batch['question'].to(DEVICE)
        label = batch['label'].to(DEVICE)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        ):
            logits = model(images, questions)
            loss   = compute_loss(logits, label)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
    
    return total_loss / len(loader)


@torch.no_grad()
def val_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    pbar = tqdm(loader, desc="Validation")
    
    for batch in pbar:
        images = batch['image'].to(DEVICE)
        questions = batch['question'].to(DEVICE)
        label = batch['label'].to(DEVICE)
        
        logits = model(images, questions)
        loss = compute_loss(logits, label)
        
        total_loss += loss.item()
        total_acc += compute_acc(logits, label)
        pbar.set_postfix(loss=loss.item(), avg_loss=total_loss / (pbar.n + 1))
    
    return total_loss / len(loader), total_acc / len(loader.dataset)


def train(
    model,
    train_loader,
    val_loader,
    n_epochs: int,
    lr: float,
    patience: int,
    checkpoint_path: Path
):
    scaler = torch.amp.GradScaler("cuda")
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_val_acc = 0.0
    epochs_without_improvement = 0

    for epoch in range(n_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, scaler)
        val_loss, val_acc = val_epoch(model, val_loader)

        scheduler.step(val_acc)
        current_lr = optimizer.param_groups[0]['lr']

        print(
            f"Epoch {epoch+1}/{n_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc*100:.2f}% | "
            f"LR: {current_lr:.2e}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0

            state_dict = (
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )

            torch.save(state_dict, checkpoint_path)
            print(f"✓ New best model saved (val_acc={best_val_acc*100:.2f}%)")

        else:
            epochs_without_improvement += 1
            print(f"No improvement for {epochs_without_improvement}/{patience} epochs")


        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping triggered after {patience} epochs without improvement.")
            break

    if checkpoint_path.exists():
        state_dict = torch.load(checkpoint_path, map_location=DEVICE)

        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(state_dict)
        else:
            model.load_state_dict(state_dict)

    return model

# VQA

In [ ]:
def start_vqa_train(n_epochs: int = 30, lr: float = 5e-4, patience: int = 3):
    print('Loading VQA data')
        
    vqa_train = load_vqa_data(VQA_QUESTIONS_TRAIN_PATH, VQA_ANNOTATIONS_TRAIN_PATH)
    vqa_val = load_vqa_data(VQA_QUESTIONS_VAL_PATH, VQA_ANNOTATIONS_VAL_PATH)
        
    vqa_images_train = torch.load(VQA_IMAGES_FEAT_TRAIN_PATH, map_location='cpu')
    vqa_images_val = torch.load(VQA_IMAGES_FEAT_VAL_PATH, map_location='cpu')
        
    vqa_vocab = build_answers_vocab(vqa_train, VQA_ANSWER_VOCAB_SIZE)
        
    vqa_tokenizer = QuestionTokenizer(max_size=QUESTION_VOCAB_MAX_SIZE, min_freq=5, special_tokens=['<PAD>', '<UNK>'])
    vqa_tokenizer.build_vocab(vqa_train)
    
    vqa_train = vqa_tokenizer.encode_questions(vqa_train, MAX_SEQ_LEN, True)
    vqa_val = vqa_tokenizer.encode_questions(vqa_val, MAX_SEQ_LEN, True)
        
    vqa_train = compute_vqa_soft_labels(vqa_train, vqa_vocab)
    vqa_val = compute_vqa_soft_labels(vqa_val, vqa_vocab)
    
    print('Training on VQA data')
        
    batch_size = 256
        
    vqa_train_dataset = HWpDataset(vqa_train, vqa_images_train)
    vqa_val_dataset = HWpDataset(vqa_val, vqa_images_val)
        
    vqa_train_loader = DataLoader(
        vqa_train_dataset,
        batch_size=batch_size,
        shuffle=True, 
        num_workers=4, 
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )
    vqa_val_loader = DataLoader(
        vqa_val_dataset, 
        batch_size=batch_size,
        shuffle=False,
        num_workers=4, 
        pin_memory=True,
        persistent_workers=True, 
        prefetch_factor=2
    )
    
    glove_weights = build_glove_weights(vqa_tokenizer, glove, 300)
            
    model = VanillaVQA(
        answer_vocab_size=len(vqa_vocab),
        question_vocab_size=len(vqa_tokenizer.word2idx),
        dropout=0.15,
        glove_weights=glove_weights
    )
        
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPU")
        model = nn.DataParallel(model)
        
    model = model.to(DEVICE)
        
    model = train(
        model=model,
        train_loader=vqa_train_loader,
        val_loader=vqa_val_loader,
        n_epochs=n_epochs,
        lr=lr,
        patience=patience,
        checkpoint_path=Path("vqa_best_model.pt")
    )

    return model, vqa_vocab, vqa_tokenizer


@torch.no_grad()
def eval_vqa(
    model,
    questions_path: Path,
    image_feats_path: Path,
    vqa_vocab,
    tokenizer: QuestionTokenizer,
    output_path: Path,
):
    model.eval()

    vqa_test = load_vqa_test_data(questions_path)
    vqa_test = tokenizer.encode_questions(vqa_test, MAX_SEQ_LEN, True)
    vqa_image_features_test = torch.load(image_feats_path, map_location='cpu')

    test_dataset = HWpDataset(vqa_test, vqa_image_features_test)

    test_loader = DataLoader(
        test_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )

    idx2answer = {idx: ans for ans, idx in vqa_vocab.items()}
    predictions = []

    for batch in tqdm(test_loader, desc="Generating predictions"):
        images = batch["image"].to(DEVICE)
        questions = batch["question"].to(DEVICE)

        logits = model(images, questions)
        preds = logits.argmax(dim=-1)

        for question_id, pred_idx in zip(batch["question_id"], preds):
            answer = idx2answer[pred_idx.item()]

            predictions.append({
                "question_id": int(question_id),
                "answer": answer
            })

    with open(output_path, "w") as f:
        json.dump(predictions, f)

    print(f"Saved {len(predictions)} predictions to {output_path}")

In [ ]:
vqa_model, vqa_vocab, vqa_tokenizer = start_vqa_train(30, 5e-4, 3)
eval_vqa(vqa_model, VQA_QUESTIONS_TEST_PATH, VQA_IMAGES_FEAT_TEST_PATH, vqa_vocab, vqa_tokenizer, Path("vqa_predictions_test.json"))

# GQA

In [ ]:
def start_gqa_train(n_epochs: int = 30, lr: float = 5e-4, patience: int = 3):
    print('Loading GQA data')

    gqa_train = load_gqa_data(GQA_QUESTIONS_TRAIN_PATH)
    gqa_val = load_gqa_data(GQA_QUESTIONS_VAL_PATH)
    gqa_images = torch.load(GQA_IMAGES_FEAT_PATH, map_location='cpu')
    
    gqa_vocab = build_answers_vocab(gqa_train, GQA_ANSWER_VOCAB_SIZE)
    
    gqa_tokenizer = QuestionTokenizer(max_size=QUESTION_VOCAB_MAX_SIZE, min_freq=5, special_tokens=['<PAD>', '<UNK>'])
    gqa_tokenizer.build_vocab(gqa_train)

    gqa_train = gqa_tokenizer.encode_questions(gqa_train, MAX_SEQ_LEN, True)
    gqa_val = gqa_tokenizer.encode_questions(gqa_val, MAX_SEQ_LEN, True)

    gqa_train = compute_gqa_label(gqa_train, gqa_vocab)
    gqa_val = compute_gqa_label(gqa_val, gqa_vocab)
    
    print('Training on GQA data')
    
    batch_size = 256
    glove_weights = build_glove_weights(gqa_tokenizer, glove, 300)
    
    gqa_train_dataset = HWpDataset(gqa_train, gqa_images)
    gqa_val_dataset = HWpDataset(gqa_val, gqa_images)

    gqa_train_loader = DataLoader(
        gqa_train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True, 
        persistent_workers=True,
        prefetch_factor=2
    )
    gqa_val_loader = DataLoader(
        gqa_val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=4, 
        pin_memory=True,
        persistent_workers=True, 
        prefetch_factor=2
    )

    model = VanillaVQA(
        answer_vocab_size=len(gqa_vocab),
        question_vocab_size=len(gqa_tokenizer.word2idx),
        dropout=0.5,
        glove_weights=glove_weights
    )
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPU")
        model = nn.DataParallel(model)
    
    model = model.to(DEVICE)
    
    model = train(
        model=model,
        train_loader=gqa_train_loader,
        val_loader=gqa_val_loader,
        n_epochs=n_epochs,
        lr=lr,
        patience=patience,
        checkpoint_path=Path("gqa_best_model.pt")
    )

    return model, gqa_vocab, gqa_tokenizer


@torch.no_grad()
def eval_gqa(
    model,
    questions_path: Path,
    image_feats_path: Path,
    gqa_vocab,
    tokenizer: QuestionTokenizer,
    output_path,
):
    model.eval()

    gqa_questions_test = load_gqa_data(questions_path)
    gqa_questions_test = tokenizer.encode_questions(gqa_questions_test, MAX_SEQ_LEN, True)
    gqa_questions_test = compute_gqa_label(gqa_questions_test, gqa_vocab)

    gqa_image_features_test = torch.load(image_feats_path, map_location="cpu")

    test_dataset = HWpDataset(gqa_questions_test, gqa_image_features_test)

    test_loader = DataLoader(
        test_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2
    )

    # Metrics
    binary_correct = 0
    binary_total = 0

    other_correct = 0
    other_total = 0

    total_correct = 0
    total = 0

    for batch in tqdm(test_loader, desc="Computing GQA stats"):
        images = batch["image"].to(DEVICE)
        questions = batch["question"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        logits = model(images, questions)

        preds = logits.argmax(dim=-1)

        correct_mask = preds == labels

        total_correct += correct_mask.sum().item()
        total += labels.size(0)

        is_binary = batch["is_binary"].bool().to(DEVICE)

        if is_binary.any():
            binary_correct += correct_mask[is_binary].sum().item()
            binary_total += is_binary.sum().item()

        other_mask = ~is_binary

        if other_mask.any():
            other_correct += correct_mask[other_mask].sum().item()
            other_total += other_mask.sum().item()

    overall_acc = total_correct / total

    binary_acc = (
        binary_correct / binary_total
        if binary_total > 0 else 0
    )

    other_acc = (
        other_correct / other_total
        if other_total > 0 else 0
    )

    print(f"Overall Accuracy: {overall_acc:.4f}")
    print(f"Binary Accuracy: {binary_acc:.4f}")
    print(f"Other Accuracy: {other_acc:.4f}")

    with open(output_path, "w") as f:
        json.dump({
            "overall": overall_acc,
            "binary": binary_acc,
            "other": other_acc
        }, f)

    print(f"Saved GQA stats to {output_path}")

In [ ]:
gqa_model, gqa_vocab, gqa_tokenizer = start_vqa_train(30, 5e-4, 3)
eval_vqa(gqa_model, GQA_QUESTIONS_TESTDEV_PATH, GQA_IMAGES_FEAT_PATH, gqa_vocab, gqa_tokenizer, Path("vanilla_gqa.json"))